## Construct VOCAB table

In [99]:
# import libraries
import pandas as pd
import numpy as np
import glob
import os
import re
import nltk
import xml.etree.ElementTree as ET
from pathlib import Path
from bs4 import BeautifulSoup
from nltk.stem.porter import PorterStemmer

In [100]:
# load in the corpus
OHCO = ['book_id', 'chap_num', 'para_num', 'sent_num', 'token_num']
CORPUS = pd.read_csv('data/CORPUS.csv', sep='\t').set_index(OHCO)

### Extract VOCAB

Features:
- n, p, i, n_chars

In [101]:
# extract VOCAB from CORPUS and get counts
VOCAB = CORPUS.term_str.value_counts().to_frame('n').sort_index()

# derive number of characters in terms
VOCAB['n_chars'] = VOCAB.index.str.len()

# derive probability (likelihood of a token appearing in the text)
VOCAB['p'] = VOCAB.n / VOCAB.n.sum()

# derive information (how surprising or informative a term is)
VOCAB['i'] = -np.log2(VOCAB.p)

### Annotate VOCAB

Features:
- max_pos, max_pos_group
- n_pos, n_pos_group (pos ambiguity)
- stop (stopwords)
- porter stem
- dfidf (gets added later in pipeline)
- add ngrams if time permits?

In [102]:
# get max POS (most frequently associated part of speech category for each word)

# max pos per term (count every pair, unstack pos tags into columns, find column name with highest count per row)
VOCAB['max_pos'] = CORPUS[['term_str', 'pos']].value_counts().unstack(fill_value=0).idxmax(1)

# assign term a max pos group
VOCAB['max_pos_group'] = CORPUS[['term_str','pos_group']].value_counts().unstack(fill_value=0).idxmax(1)

In [103]:
# compute POS ambiguity (how many pos, by group, are associated with each term in the corpus)

# 
VOCAB['n_pos_group'] = CORPUS[['term_str','pos_group']].value_counts().unstack().count(1)
# VOCAB['cat_pos_group'] = CORPUS[['term_str','pos_group']].value_counts().to_frame('n').reset_index()\
#     .groupby('term_str').pos_group.apply(lambda x: set(x))
VOCAB['n_pos'] = CORPUS[['term_str','pos']].value_counts().unstack().count(1)
# VOCAB['cat_pos'] = CORPUS[['term_str','pos']].value_counts().to_frame('n').reset_index()\
#     .groupby('term_str').pos.apply(lambda x: set(x))

# TODO FIX REDUNDANT COMPUTATION?

In [104]:
VOCAB.sample(10)

,n,n_chars,p,i,max_pos,max_pos_group,n_pos_group,n_pos
term_str,,,,,,,,
stifled,5,7,0.000006,17.378574,JJ,JJ,2,2
surnames,3,8,0.000004,18.115540,NNS,NN,1,1
catechism,1,9,0.000001,19.700502,NN,NN,1,1
changes,8,7,0.000009,16.700502,NNS,NN,2,2
erroneous,2,9,0.000002,18.700502,JJ,JJ,1,1
reassuringly,10,12,0.000012,16.378574,RB,RB,1,1
park,76,4,0.000089,13.452575,NNP,NN,2,3
squalid,3,7,0.000004,18.115540,JJ,JJ,1,1
bounds,3,6,0.000004,18.115540,NNS,NN,1,1


In [105]:
VOCAB.loc[['s','t','don','ve','ll']]

,n,n_chars,p,i,max_pos,max_pos_group,n_pos_group,n_pos
term_str,,,,,,,,
s,8006,1,0.009397,6.733636,NN,VB,11,19
t,5741,1,0.006738,7.213416,NN,NN,8,13
don,1791,3,0.002102,8.893953,VBP,VB,3,6
ve,1371,2,0.001609,9.279490,JJ,JJ,6,8
ll,906,2,0.001063,9.877135,JJ,JJ,7,13


### Add Stopwords

Using NLTK's built in list for English and choosing to add 'said' to stopword list as it functions mostly as a dialogue marker rather than a content-bearing term.

In [106]:
# get stopwords from NLTK's built in list for English
sw = set(nltk.corpus.stopwords.words('english'))
# add dialogue marker to stopwords
sw.update(['said']) # use .update() to add elements from an iterable into an existing set in place
VOCAB['stop'] = VOCAB.index.isin(sw)

In [107]:
# check high freq terms not flagged as stopwords
VOCAB[VOCAB.stop == False].sort_values('n', ascending=False).head(50)

,n,n_chars,p,i,max_pos,max_pos_group,n_pos_group,n_pos,stop
term_str,,,,,,,,,
one,2963,3,0.003478,8.167659,CD,CD,6,10,False
poirot,2961,6,0.003475,8.168634,NNP,NN,2,4,False
know,2600,4,0.003052,8.356206,VBP,VB,4,9,False
would,2321,5,0.002724,8.519972,MD,MD,4,5,False
man,2090,3,0.002453,8.671215,NN,NN,2,3,False
think,1870,5,0.002195,8.831680,VBP,VB,4,7,False
mr,1861,2,0.002184,8.838640,NNP,NN,1,1,False
see,1833,3,0.002151,8.860511,VB,VB,3,5,False
well,1830,4,0.002148,8.862874,RB,RB,7,10,False


In [108]:
# check flagged stopwords for things that shouldn't be in stopwords

In [109]:
VOCAB[VOCAB.stop == True].sort_values('n', ascending=False).head(30)

,n,n_chars,p,i,max_pos,max_pos_group,n_pos_group,n_pos,stop
term_str,,,,,,,,,
the,38116,3,0.044737,4.482393,DT,DT,6,9,True
i,25846,1,0.030336,5.042849,PRP,PR,4,5,True
to,21600,2,0.025352,5.301759,TO,TO,4,8,True
a,19626,1,0.023035,5.440024,DT,DT,5,7,True
and,18320,3,0.021502,5.539370,CC,CC,6,10,True
of,17495,2,0.020534,5.605847,IN,IN,6,8,True
you,15233,3,0.017879,5.805590,PRP,PR,7,13,True
it,13923,2,0.016341,5.935320,PRP,PR,6,9,True
he,13655,2,0.016027,5.963361,PRP,PR,4,5,True


In [110]:
VOCAB[VOCAB.stop == 1]

,n,n_chars,p,i,max_pos,max_pos_group,n_pos_group,n_pos,stop
term_str,,,,,,,,,
a,19626,1,0.023035,5.440024,DT,DT,5,7,True
about,2441,5,0.002865,8.447246,IN,IN,5,6,True
above,63,5,0.000074,13.723222,IN,IN,4,4,True
after,999,5,0.001173,9.736162,IN,IN,3,4,True
again,1004,5,0.001178,9.728959,RB,RB,5,9,True
...,...,...,...,...,...,...,...,...,...
you,15233,3,0.017879,5.805590,PRP,PR,7,13,True
your,1927,4,0.002262,8.788362,PRP$,PR,3,4,True
yours,161,5,0.000189,12.369585,NNS,NN,5,9,True


In [111]:
print("Stopwords:", int(VOCAB.stop.sum()))

Stopwords: 154


### Add Porter Stem

In [112]:
# instantiate stemmer (outside the lambda function)
porter = PorterStemmer()

VOCAB['porter_stem'] = VOCAB.index.map(lambda x: porter.stem(x))

In [113]:
VOCAB

,n,n_chars,p,i,max_pos,max_pos_group,n_pos_group,n_pos,stop,porter_stem
term_str,,,,,,,,,,
1,33,1,0.000039,14.656108,CD,CD,1,1,False,1
10,2,2,0.000002,18.700502,CD,CD,1,1,False,10
100,2,3,0.000002,18.700502,CD,VB,2,2,False,100
100000,1,6,0.000001,19.700502,NN,NN,1,1,False,100000
1015,1,4,0.000001,19.700502,CD,CD,1,1,False,1015
...,...,...,...,...,...,...,...,...,...,...
évidemment,3,10,0.000004,18.115540,NN,NN,1,1,False,évidem
évident,1,7,0.000001,19.700502,NN,NN,1,1,False,évident
êtes,1,4,0.000001,19.700502,NNS,NN,1,1,False,ête


In [114]:
# check
VOCAB.columns.tolist()

['n',
 'n_chars',
 'p',
 'i',
 'max_pos',
 'max_pos_group',
 'n_pos_group',
 'n_pos',
 'stop',
 'porter_stem']

In [115]:
# save the VOCAB table to csv
VOCAB.to_csv('data/VOCAB.csv', sep='\t', index=True)